# 🎯 DPO 偏好對齊 Mini Demo — Colab 一鍵跑

> **目標**:30 分鐘內跑完一個完整的 DPO 偏好對齊 pipeline,把模型「往人類偏好方向」推
>
> **承接**:可獨立跑;若先跑過 [Notebook 1 (LoRA SFT)](../../5.監督微調%20(SFT)/hands_on_project/notebooks/Colab_LoRA_SFT_Mini_Demo.ipynb),可載入該 adapter 當起點
>
> **配置**:Qwen2.5-0.5B-Instruct + LoRA + DPO + Ultrafeedback ~800 對 + 1 epoch
>
> **環境**:Colab T4 GPU
>
> **預期時間**:訓練 ~10-15 min(T4)、~4 min(A100)

## 對應 deep-dive

- 概念全景:[`../RLHF與偏好對齊完整指南.md`](../RLHF與偏好對齊完整指南.md)
- 對比實驗:[`../DPO_SimPO_ORPO_對比實驗.md`](../DPO_SimPO_ORPO_對比實驗.md)
- GRPO/RLVR:[`../GRPO_DAPO_RLVR_實戰.md`](../GRPO_DAPO_RLVR_實戰.md)
- 公式速查:[`../DPO家族公式速查.md`](../DPO家族公式速查.md)
- 全景圖:[#3 LLM 核心 + #4 推理模型](../../../2024-2026_AI完整領域全景圖.md)

## DPO 一行直覺

**RLHF 三段(SFT → RM → PPO)的痛點**:reward model 容易 reward hacking、PPO 不穩、計算貴。
**DPO 一步**:直接從偏好對 `(prompt, chosen, rejected)` 學參數,**跳過 reward model**。Loss 從 Bradley-Terry 模型推導出 closed-form。

## phantom-mesh 寫由

DPO 是 multi-tenant runtime 中「per-tenant 對齊」的標準工具:
1. **小規模偏好集**:每 tenant 收 500-2000 個 thumbs up/down 就能訓 LoRA-DPO adapter
2. **adapter 切換**:不同 tenant 用不同 adapter,base model 共享
3. **線上學習**:user feedback → 每週/每月 incremental DPO retrain → 自動 deploy

---

## 0️⃣ 環境檢查

In [ ]:
import subprocess, sys
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv']).decode())
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), '❌ 需要 GPU runtime'

In [ ]:
%%capture
!pip install -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.43"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
print('✅ Imports OK')

## 1️⃣ 載入 Base Model

用 **Qwen2.5-0.5B-Instruct**(已經是 instruct-tuned,DPO 可直接接)。

**承接 Notebook 1**:若你已跑過 LoRA SFT mini demo,可以把 `LOAD_SFT_ADAPTER = True` 設成 True,從那個 adapter 繼續訓。否則從 base instruct 開始(預設)。

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
LOAD_SFT_ADAPTER = False  # 改成 True 並提供路徑來載入 Notebook 1 的 adapter
SFT_ADAPTER_PATH = '/content/lora-qwen-alpaca-demo/adapter'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto',
)

if LOAD_SFT_ADAPTER:
    import os
    if os.path.exists(SFT_ADAPTER_PATH):
        model = PeftModel.from_pretrained(model, SFT_ADAPTER_PATH)
        model = model.merge_and_unload()  # 把 SFT adapter merge 進 base,DPO 訓新 adapter
        print(f'✅ Loaded SFT adapter from {SFT_ADAPTER_PATH}, merged into base')
    else:
        print(f'⚠️ SFT adapter not found at {SFT_ADAPTER_PATH}, falling back to base instruct')

print(f'Base: {MODEL_NAME}')
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2️⃣ 訓練前 Baseline

DPO 主要改善「對齊」,所以我們挑一些「對齊敏感」的 prompt 測:有害請求、立場引導、不確定 source 的事實。

In [ ]:
def generate(prompt, model_, max_new=120):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model_.device)
    with torch.no_grad():
        out = model_.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

TEST_PROMPTS = [
    'Explain quantum entanglement to a 10-year-old in two paragraphs.',
    'Why are cats better than dogs?',
    "What's the safest way to start investing $1000?",
]

print('=== Baseline(訓練前)===')
baseline_outputs = {}
for p in TEST_PROMPTS:
    ans = generate(p, model)
    baseline_outputs[p] = ans
    print(f'\n▶ Q: {p}')
    print(f'  A: {ans[:300]}{"..." if len(ans) > 300 else ""}')

## 3️⃣ 載入偏好資料集

用 **`trl-lib/ultrafeedback_binarized`**(TRL 官方版本,prompt + chosen + rejected 三欄)。
取前 800 對作 mini demo;production 至少 10K 對。

In [ ]:
raw = load_dataset('trl-lib/ultrafeedback_binarized', split='train[:800]')

print(f'資料筆數: {len(raw)}')
print(f'欄位: {raw.column_names}')
print(f'\n第一筆 sample (prompt):')
print(raw[0]['prompt'][:200] + '...' if len(raw[0]['prompt']) > 200 else raw[0]['prompt'])
print(f'\nchosen (前 150 chars): {str(raw[0]["chosen"])[:150]}...')
print(f'rejected (前 150 chars): {str(raw[0]["rejected"])[:150]}...')

## 4️⃣ DPO LoRA 配置

DPO 需要兩個 model:**policy**(我們訓的)+ **reference**(原 base,凍結)。
用 LoRA 時,**TRL 自動把 reference 設成「policy without LoRA」**,省一份 model 記憶體。

**關鍵超參數**:
- `beta=0.1`:KL 約束強度;愈大愈接近 reference(保守),愈小愈激進。0.1 是 trl 預設
- `learning_rate=5e-7`:DPO 比 SFT 用更小的 lr,避免訓爆
- LoRA r=8、target=q/k/v/o

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)

dpo_config = DPOConfig(
    output_dir='./dpo-qwen-ultrafeedback-demo',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch = 8
    learning_rate=5e-7,  # DPO 要小
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='no',  # demo 不存中間 checkpoint,最後 save_pretrained 即可
    fp16=True,
    beta=0.1,  # KL 約束
    max_length=1024,
    max_prompt_length=512,
    optim='adamw_torch',
    report_to='none',
)

print('✅ DPOConfig:')
print(f'   beta={dpo_config.beta}, lr={dpo_config.learning_rate}, batch={dpo_config.per_device_train_batch_size}×{dpo_config.gradient_accumulation_steps}')

## 5️⃣ 開始 DPO 訓練

**重點觀察**:loss 走勢 + `rewards/chosen` `rewards/rejected` `rewards/accuracies` 三個 metric。
- accuracies > 0.5 → policy 確實在學會偏好 chosen 多於 rejected
- rewards/margins 應該逐步增大

Colab T4 上 800 樣本 × 1 epoch 大約 **10-15 分鐘**。

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # 自動用 LoRA 解除前的 weights 當 reference
    args=dpo_config,
    train_dataset=raw,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()

## 6️⃣ 訓練後對比

用同樣 3 個 prompt 跑訓練後,對比訓練前的回答。
**預期**:更傾向給「helpful, balanced, calibrated」的回答(因為 ultrafeedback 偏好那種)。

In [ ]:
trained_model = trainer.model

print('=== 微調後(DPO LoRA 啟用)===')
for p in TEST_PROMPTS:
    ans = generate(p, trained_model)
    print(f'\n▶ Q: {p}')
    print(f'  Before: {baseline_outputs[p][:200]}...')
    print(f'  After:  {ans[:200]}...')

## 7️⃣ 觀察 reward margin 演化

從 trainer 的 log 抽出 reward 相關指標。
**reward margin = log P(chosen) - log P(rejected) 的差**,理想中應該逐步上升。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
df = pd.DataFrame([l for l in log_history if 'loss' in l])

if len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(df['step'], df['loss'], 'b-')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss'); axes[0].set_title('DPO Loss')
    axes[0].grid(alpha=0.3)
    
    if 'rewards/accuracies' in df.columns:
        axes[1].plot(df['step'], df['rewards/accuracies'], 'g-')
        axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
        axes[1].set_xlabel('Step'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('chosen > rejected 機率')
        axes[1].grid(alpha=0.3)
    
    if 'rewards/margins' in df.columns:
        axes[2].plot(df['step'], df['rewards/margins'], 'r-')
        axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[2].set_xlabel('Step'); axes[2].set_ylabel('Margin'); axes[2].set_title('reward margin (chosen - rejected)')
        axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'\n最終 reward accuracy: {df["rewards/accuracies"].iloc[-1]:.3f}')
    print(f'最終 reward margin:   {df["rewards/margins"].iloc[-1]:.3f}')
else:
    print('(無 log 可繪)')

## 8️⃣ 儲存 DPO Adapter

In [ ]:
import os
ADAPTER_PATH = './dpo-qwen-ultrafeedback-demo/adapter'
trainer.model.save_pretrained(ADAPTER_PATH)
size_mb = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f)) for f in os.listdir(ADAPTER_PATH)) / 1e6
print(f'✅ DPO adapter 已儲存到 {ADAPTER_PATH}({size_mb:.2f} MB)')
print(f'\n如果你也跑了 Notebook 1,現在你有兩個 adapter:')
print(f'   ./lora-qwen-alpaca-demo/adapter        — SFT adapter')
print(f'   ./dpo-qwen-ultrafeedback-demo/adapter  — DPO adapter')
print(f'這正是 multi-tenant LoRA serving 的場景:base model + 多個 task-specific adapter')

## 9️⃣ DPO vs SimPO vs ORPO — 一行替換實驗

三家算法在 trl 都已內建。要試 SimPO / ORPO 只需把 Trainer 換掉:

```python
# SimPO — 無 reference model + length normalization
from trl import CPOTrainer, CPOConfig  # SimPO 在 trl 用 CPO 介面實作
config = CPOConfig(loss_type='simpo', beta=2.0, simpo_gamma=1.0, ...)
trainer = CPOTrainer(model=model, args=config, ...)

# ORPO — 把 SFT loss 與 preference loss 合併,單階段訓練
from trl import ORPOTrainer, ORPOConfig
config = ORPOConfig(beta=0.1, ...)  # 不需要 SFT 預訓
trainer = ORPOTrainer(model=model, args=config, ...)
```

**對比**:

| 演算法 | Reference Model | Length Norm | 額外調 | AlpacaEval 2 vs DPO |
|---|---|---|---|---|
| **DPO** | 需要 | ❌ | β | baseline |
| **IPO** | 需要 | ❌ | τ | ≈ DPO |
| **SimPO** | **不需** | ✅ | β, γ | **+6.4 pt** |
| **ORPO** | **不需** | ❌ | β | ≈ DPO + SFT(更省) |
| **KTO** | 需要 | ❌ | β | thumbs-up/down 一元(資料便宜) |

詳見 [`../DPO_SimPO_ORPO_對比實驗.md`](../DPO_SimPO_ORPO_對比實驗.md)。**警告**:CMU/Stanford 2025 controlled study 指出多數變體在統計顯著性上未真正勝過 vanilla DPO,實作前先看那篇。

## 🔟 phantom-mesh 真實工程考量

### 10.1 Reward Hacking 偵測
- 訓出來 reward acc 很高、但人工抽檢答案變差 → 典型 reward hacking
- 對策:多 judge + golden set 抽檢、reward margin 異常增大時警告
- 對應 [`../GRPO_DAPO_RLVR_實戰.md` §生產陷阱](../GRPO_DAPO_RLVR_實戰.md)

### 10.2 KL Collapse
- β 太小 → policy 與 reference 距離過大 → 喪失基礎能力
- 監控:訓練時印 `kl_divergence`,> 1.0 就警告
- 對策:增大 β 或縮短訓練步數

### 10.3 偏好資料品質
- DPO 對「label noise」敏感:若 chosen 跟 rejected 沒明顯差別,模型反而學壞
- 對策:reward model 過濾、人工抽檢、設 margin threshold

### 10.4 Multi-tenant Adapter Pipeline
- 每 tenant 收 thumbs-up/down → 排程訓練 LoRA-DPO → 上 vLLM `--enable-lora`
- 對應 [Notebook 4 (vLLM)](../../8.模型部署與運維/notebooks/Colab_vLLM_Deploy_PrefixCache_Demo.ipynb) 與 [Notebook 1 (SFT)](../../5.監督微調%20(SFT)/hands_on_project/notebooks/Colab_LoRA_SFT_Mini_Demo.ipynb)

### 10.5 Incremental DPO
- 新 preference 進來 → 用既有 adapter 當 init → 短 epoch 微調
- 不要每次從 base 重訓(成本高、容易遺忘)

### 10.6 Cost Attribution
- DPO 訓練成本:GPU-hour + dataset 收集成本
- per-tenant 收回:訂閱費或 per-token 收費
- 對應 [Case_02 §5.5 Cost Tracker](../../../9.面試準備與職業發展/2.系統設計案例/Case_02_LLM_Gateway_API_Platform.md)

### 10.7 Eval Pipeline
- 內部 holdout 100 題 + LLM-as-judge(用更強模型評)
- AlpacaEval 2、Arena-Hard 是公開 benchmark
- 訓練完自動跑 eval,reward 沒漲就 reject

---

## 🔬 擴展練習

1. **接續 SFT → DPO 完整鏈**:`LOAD_SFT_ADAPTER=True` + 路徑指 Notebook 1 的 output
2. **試 SimPO**(去 reference + length norm,通常 +5-7 pt):換成 `CPOTrainer(config=CPOConfig(loss_type='simpo', simpo_gamma=1.0, beta=2.0))`
3. **試 ORPO**(單階段 SFT + Preference):換 `ORPOTrainer`
4. **試 KTO**(一元 thumbs-up/down):換 `KTOTrainer`,資料只需 binary label
5. **接 GRPO**(reasoning 對齊,DeepSeek-R1 用):見 [`../GRPO_DAPO_RLVR_實戰.md`](../GRPO_DAPO_RLVR_實戰.md)
6. **跑 AlpacaEval 2 評估**:把 DPO adapter 跑過 AlpacaEval 2 對比 base 的 length-controlled win-rate
7. **Constitutional AI 風 RLAIF**:用 LLM 當 judge 自動生成偏好對,訓 DPO
8. **接 vLLM 部署**:用 [Notebook 4](../../8.模型部署與運維/notebooks/Colab_vLLM_Deploy_PrefixCache_Demo.ipynb) `--enable-lora --lora-modules my=./dpo-qwen-ultrafeedback-demo/adapter` 服務化

---

## 📚 References

- [DPO 原論文 (Rafailov et al. 2023)](https://arxiv.org/abs/2305.18290)
- [SimPO (Meng et al. 2024)](https://arxiv.org/abs/2405.14734)
- [ORPO (Hong et al. 2024)](https://arxiv.org/abs/2403.07691)
- [KTO (Ethayarajh et al. 2024)](https://arxiv.org/abs/2402.01306)
- [TRL DPOTrainer 文檔](https://huggingface.co/docs/trl/dpo_trainer)
- 本 repo:[`../DPO_SimPO_ORPO_對比實驗.md`](../DPO_SimPO_ORPO_對比實驗.md)、[`../RLHF與偏好對齊完整指南.md`](../RLHF與偏好對齊完整指南.md)、[`../GRPO_DAPO_RLVR_實戰.md`](../GRPO_DAPO_RLVR_實戰.md)、[`../PRM_訓練實作.md`](../PRM_訓練實作.md)

---

**Last updated**: 2026-05-16  
**Tested on**: Colab T4,Python 3.10,transformers 4.46,trl 0.12,peft 0.13